In [69]:
import numpy as np, pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score,
                             recall_score, f1_score)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

SEED = 42
DATA_PATH = "nscg2023_onet30_2_subset.csv"

**Part 1: Setup**
JOBSATIS
*   1 = very satisfied
*   2 = somewhat satisfied
*   3 = somewhat dissatisfied
*   4 = very dissatisfied


Based on these values, we changed them into a yes/no target: satisfied = 1 or 2 and unsatisfied = 3 or 4.

The occupation code is being kept as groups to make sure the occupations are together.


In [70]:
df = pd.read_csv(DATA_PATH, low_memory=False)
print("shape:", df.shape, "| missing values:", int(df.isna().sum().sum()))
print(df["JOBSATIS"].value_counts().sort_index().to_dict())

y = df["JOBSATIS"].isin([1, 2]).astype(int).values   # 1 = satisfied
groups = df["SOC2018_CODE"].values                    # one group per occupation
print(f"satisfied: {y.mean():.1%} | unsatisfied: {1 - y.mean():.1%} | occupations: {len(np.unique(groups))}")

shape: (6849, 42) | missing values: 0
{1: 3012, 2: 3218, 3: 511, 4: 108}
satisfied: 91.0% | unsatisfied: 9.0% | occupations: 25


**Part 2: Baseline Features**

For this forest, we only used background features:
*   demographics = sex, age, and degree level
*   background = demographics and work context (how they are related to job satisfaction)



In [71]:
NUM = ["AGE"]
CAT_DEMO = ["SEX_2023", "DGRDG"]
CAT_WORK = ["OCEDRLP", "SUPWK", "WAPRI"]
FEATURE_SETS = {
    "demographics_only": NUM + CAT_DEMO,
    "background_full":   NUM + CAT_DEMO + CAT_WORK,
}

# Ensures no fit features may leak into the baseline
for cols in FEATURE_SETS.values():
    assert not any(c.startswith(("FAC", "ONET_WV")) for c in cols)

In [72]:
# DGRDG is a code, not a quantity: force it to be treated as categorical
df["DGRDG"] = df["DGRDG"].astype(str)

def make_forest(cols, leaf=50):
    cat = [c for c in cols if c not in NUM]
    pre = ColumnTransformer([("num", "passthrough", [c for c in cols if c in NUM]),
                             ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
    rf = RandomForestClassifier(n_estimators=500, min_samples_leaf=leaf, max_features="sqrt",
                                class_weight="balanced_subsample", n_jobs=-1, random_state=SEED)
    return Pipeline([("pre", pre), ("clf", rf)])

**Part 3: Cross Validation Folds**

This splits the people into 5 parts; each part takes a turn as the test set while the model trains on the other four. People in the same occupation are similar, so we group by occupation.


In [73]:
if Path("folds.csv").exists():
    saved = pd.read_csv("folds.csv")
    fold_id = df[["OBSNUM"]].merge(saved, on="OBSNUM", how="left", validate="one_to_one")["fold"].values
    assert not np.isnan(fold_id).any(), "folds.csv does not cover every respondent"
    fold_id = fold_id.astype(int)
    folds = [(np.where(fold_id != k)[0], np.where(fold_id == k)[0]) for k in range(5)]
    print("loaded folds.csv")
else:
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    folds = list(cv.split(df, y, groups))
    fold_id = np.empty(len(df), dtype=int)
    for k, (_, te) in enumerate(folds):
        fold_id[te] = k
    pd.DataFrame({"OBSNUM": df["OBSNUM"], "fold": fold_id}).to_csv("folds.csv", index=False)
    print("created folds.csv")

print("fold sizes:      ", [len(te) for _, te in folds])
print("unsatisfied/fold:", [int((1 - y[te]).sum()) for _, te in folds])
print("occupations/fold:", [len(np.unique(groups[te])) for _, te in folds])

loaded folds.csv
fold sizes:       [789, 1873, 1361, 1138, 1688]
unsatisfied/fold: [67, 193, 130, 101, 128]
occupations/fold: [5, 4, 5, 5, 6]


Part 4: Metrics

Unsatisfied people are rare, so plain accuracy would look good even for a model that predicts ‘satisfied’ for everyone. Instead we use:
*   **ROC AUC:** the chance that a random satisfied person gets a higher "satisfied" score than a random unsatisfied person (0.5 = coin flip, 1.0 = perfect).
*   **PR AUC (unsatisfied class):** how well the model finds the unsatisfied group without many false alarms. A model with no skill scores about 0.09, the share of unsatisfied people.
*   **Precision, recall and F1 (unsatisfied class)** at a 0.5 threshold.

In [74]:
def predict(model_fn, cols):
    """Out-of-fold P(satisfied) for every respondent."""
    p = np.zeros(len(df))
    for tr, te in folds:
        m = model_fn(cols)
        m.fit(df.iloc[tr][cols], y[tr])
        p[te] = m.predict_proba(df.iloc[te][cols])[:, 1]
    return p

def metrics(y, p, thr=0.5):
    pu, yu, pred_u = 1 - p, 1 - y, (p < thr).astype(int)    # unsatisfied = positive class
    return {"roc_auc": roc_auc_score(y, p),
            "pr_auc_unsat": average_precision_score(yu, pu),
            "precision_unsat": precision_score(yu, pred_u, zero_division=0),
            "recall_unsat": recall_score(yu, pred_u),
            "f1_unsat": f1_score(yu, pred_u)}

def cluster_boot_ci(y, p, groups, n=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    ug = np.unique(groups)
    idx = {g: np.where(groups == g)[0] for g in ug}
    out = []
    for _ in range(n):
        ii = np.concatenate([idx[g] for g in rng.choice(ug, len(ug))])
        if len(np.unique(y[ii])) == 2:
            out.append(roc_auc_score(y[ii], p[ii]))
    return np.percentile(out, [2.5, 97.5])

**Part 5: Fit and compare models**

*   **dummy_prior:** always predicts the overall share of satisfied people. This is the floor (its pooled AUC can dip below 0.5 because the share differs slightly by fold).
*   **forest_demographics_only** and **forest_background_full:** the forest on each feature set.

In [75]:
rows, preds = [], {"OBSNUM": df["OBSNUM"], "y_satisfied": y, "fold": fold_id}
dummy = lambda cols: DummyClassifier(strategy="prior")
specs = ([("dummy_prior", dummy, NUM)]
         + [(f"forest_{k}", make_forest, v) for k, v in FEATURE_SETS.items()])

for name, fn, cols in specs:
    p = predict(fn, cols)
    preds[name] = p
    r = {"model": name, **metrics(y, p)}
    r["auc_ci_low"], r["auc_ci_high"] = (cluster_boot_ci(y, p, groups) if name != "dummy_prior" else (np.nan, np.nan))
    fold_aucs = [roc_auc_score(y[te], p[te]) for _, te in folds]
    r["fold_auc_mean"], r["fold_auc_sd"] = np.mean(fold_aucs), np.std(fold_aucs)
    rows.append(r)

results = pd.DataFrame(rows).round(3)
results.to_csv("forest_baseline_results.csv", index=False)
pd.DataFrame(preds).to_csv("forest_oof_predictions.csv", index=False)
results

,model,roc_auc,pr_auc_unsat,precision_unsat,recall_unsat,f1_unsat,auc_ci_low,auc_ci_high,fold_auc_mean,fold_auc_sd
0,dummy_prior,0.465,0.084,0.000,0.000,0.000,NaN,NaN,0.500,0.000
1,forest_demographics_only,0.511,0.099,0.091,0.407,0.149,0.484,0.535,0.516,0.013
2,forest_background_full,0.586,0.124,0.124,0.459,0.195,0.557,0.617,0.590,0.030


**Part 6: Importance of Forest Settings**

This only checks whether the conclusion depends on the leaf size we chose. Do not pick the best row and report it as the baseline, because that would tune on the evaluation folds.

In [76]:
sensitivity = []
cols = FEATURE_SETS["background_full"]
for leaf in [10, 25, 50, 100, 200]:
    p = predict(lambda c, leaf=leaf: make_forest(c, leaf), cols)
    sensitivity.append({"min_samples_leaf": leaf, "roc_auc": round(roc_auc_score(y, p), 3),
                 "pr_auc_unsat": round(average_precision_score(1 - y, 1 - p), 3)})
sensitivity = pd.DataFrame(sensitivity)
sensitivity.to_csv("forest_leaf_sensitivity.csv", index=False)
sensitivity

,min_samples_leaf,roc_auc,pr_auc_unsat
0,10,0.575,0.118
1,25,0.583,0.122
2,50,0.586,0.124
3,100,0.587,0.124
4,200,0.586,0.125


**Part 7: Important Features**

For each fold we shuffle one original column in the held-out data and measure how much ROC AUC drops. A big drop means the forest relies on that feature;
a drop near zero (or negative) =  nothing. Values are averaged over the 5 folds. They show what the model uses, not what causes dissatisfaction.

In [77]:
cols = FEATURE_SETS["background_full"]
imp = np.zeros((len(folds), len(cols)))
for k, (tr, te) in enumerate(folds):
    m = make_forest(cols).fit(df.iloc[tr][cols], y[tr])
    pi = permutation_importance(m, df.iloc[te][cols], y[te], scoring="roc_auc",
                                n_repeats=10, random_state=SEED, n_jobs=-1)
    imp[k] = pi.importances_mean
importance = (pd.DataFrame({"feature": cols, "mean_auc_drop": imp.mean(axis=0), "sd_across_folds": imp.std(axis=0)})
              .sort_values("mean_auc_drop", ascending=False).round(4))
importance.to_csv("forest_permutation_importance.csv", index=False)
importance

,feature,mean_auc_drop,sd_across_folds
3,OCEDRLP,0.0583,0.0195
0,AGE,0.0329,0.0052
2,DGRDG,0.0026,0.0071
1,SEX_2023,0.0008,0.0048
4,SUPWK,0.0007,0.0049
5,WAPRI,-0.0019,0.0085
